## sequential workflows ( PROMPT CHAINING )

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [16]:
load_dotenv()

model = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

In [24]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [25]:
def create_outline(state:BlogState) -> BlogState:
    title = state['title']
    prompt = f'generate detailed report on {title}'
    outline = model.invoke(prompt).content
    state['outline']=outline
    return state
    
def create_blog(state:BlogState) -> BlogState:
    outline = state['outline']
    title = state['title']
    prompt = f"generate detailed blog on topic {title} using following blog {outline}"
    content = model.invoke(prompt).content
    state['content']=content
    return state

In [26]:
graph = StateGraph(BlogState)

# add nodes
graph.add_node("create_outline",create_outline)
graph.add_node("create_blog",create_blog)

# add edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)


In [32]:
# compile
workflow = graph.compile()

# execute
initial_state = {'title':'Rise of AI in India'}
final_state = workflow.invoke(initial_state)
print(f"outline: \n\n{final_state['outline']}\n")
print(f"\n\n content: \n\n{final_state['content']}\n")

outline: 

**Report: The Rise of AI in India**

**Executive Summary**

Artificial Intelligence (AI) has emerged as a key driver of innovation and growth in India, with the country witnessing significant advancements in AI adoption, development, and deployment across various sectors. This report highlights the current state of AI in India, its applications, challenges, and opportunities, as well as the government's initiatives to promote AI adoption.

**Introduction**

India has been rapidly embracing AI, driven by factors such as increasing digitalization, advancements in computing power, and the availability of large datasets. The country's AI ecosystem is characterized by a strong startup culture, a growing talent pool of AI professionals, and investments from both domestic and international players. This report aims to provide an in-depth analysis of the AI landscape in India, its current developments, and future prospects.

**Current State of AI in India**

1. **Growth of AI Startu